# Pipeline Financeiro 2026 — Versão Final Profissional

Notebook único para preparar, validar e exportar o dataset financeiro de 2026.

## Princípios

- dados `raw` nunca são alterados;
- cada regra de negócio é aplicada uma única vez;
- `CODACT 13` é excluído do universo financeiro;
- KILOSPO é calculado integralmente, mas apenas o que associa ao `inform_27` entra no financeiro;
- diferenças Danone são classificadas, não forçadas;
- Setembro permanece no dataset, mas é excluído da reconciliação Danone enquanto estiver incompleto;
- ocupação é calculada ao nível da viagem (`CODEUT`);
- divisões por zero devolvem `NaN`;
- a exportação só ocorre depois das validações finais.

## 00. Configuração

In [34]:
# ==========================
# 1. Imports
# ==========================
from pathlib import Path
import platform
import duckdb
import numpy as np
import pandas as pd

# ==========================
# 2. Parâmetros
# ==========================
ANO = 2026
MES_VALIDACAO_DANONE_MAX = 9

CUSTO_ESTRUTURA_MENSAL = 103_000.0
EXTRA_DANONE_MENSAL = 20_000.0
DESFASAMENTO_MES_COMBUSTIVEL = -1

CODACT_ESPANHA = [338, 211, 15]
CODACT_EXCLUIR = [13]

TOLERANCIA_FINANCEIRA = 0.01

MAPA_CAPACIDADE = {
    4: 6,
    5: 6,
    6: 6,
    8: 12,
    12: 12,
    14: 14,
    15: 16,
    16: 16,
    18: 18,
    20: 20,
    22: 22,
    24: 24,
    33: 33,
    55: 55,
    66: 66,
}

# ==========================
# 3. Caminhos
# ==========================
def get_paths():
    sistema = platform.system()

    if sistema == "Windows":
        base = Path(r"C:\Users\LISARR\OneDrive - Salvesen Logística S.A\python\01.Financeiro")
        return {
            "db": Path(r"C:\Users\LISARR\OneDrive - Salvesen Logística S.A\python\00.DB\2026.duckdb"),
            "excel": base / "Danone_Custos_V3.xlsx",
            "parquet": base / "inform_27_2026_final.parquet",
            "viagens": base / "viagens_2026.parquet",
            "validacao": base / "validacao_financeira_2026.csv",
            "danone_diag": base / "validacao_danone_2026.csv",
        }

    if sistema == "Darwin":
        base = Path(
            "/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen"
        )
        return {
            "db": base / "00_DB" / "2026.duckdb",
            "excel": base / "Danone_Custos_v3.xlsx",
            "parquet": base / "inform_27_2026_final.parquet",
            "viagens": base / "viagens_2026.parquet",
            "validacao": base / "validacao_financeira_2026.csv",
            "danone_diag": base / "validacao_danone_2026.csv",
        }

    base = Path.cwd()
    return {
        "db": base / "2026.duckdb",
        "excel": base / "Danone_Custos_v3.xlsx",
        "parquet": base / "inform_27_2026_final.parquet",
        "viagens": base / "viagens_2026.parquet",
        "validacao": base / "validacao_financeira_2026.csv",
        "danone_diag": base / "validacao_danone_2026.csv",
    }

PATHS = get_paths()


## 01. Funções comuns

In [35]:
# ==========================
# 1. Datas
# ==========================
def normalizar_data(serie):
    texto = (
        serie.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

    data = pd.to_datetime(
        texto,
        format="%Y%m%d",
        errors="coerce",
    )

    mask = data.isna()

    if mask.any():
        data.loc[mask] = pd.to_datetime(
            texto.loc[mask],
            errors="coerce",
        )

    return data

# ==========================
# 2. Referências
# ==========================
def normalizar_referencia(serie):
    return (
        serie.astype("string")
        .str.strip()
        .str.extract(r"(\d{10})", expand=False)
        .astype("string")
    )

# ==========================
# 3. Códigos postais
# ==========================
def normalizar_cp(serie):
    cp = (
        serie.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.replace(r"\D", "", regex=True)
        .str[:7]
    )

    invalido = (
        cp.str.len().fillna(0).lt(4)
        | cp.eq("0").fillna(False)
        | cp.eq("").fillna(False)
    )

    return cp.mask(invalido, pd.NA)

# ==========================
# 4. Divisão segura
# ==========================
def dividir_seguro(numerador, denominador):
    n = pd.to_numeric(numerador, errors="coerce")
    d = pd.to_numeric(denominador, errors="coerce")
    return n.div(d.where(d.ne(0)))

# ==========================
# 5. Moda segura
# ==========================
def moda_segura(serie):
    serie = serie.dropna()

    if serie.empty:
        return np.nan

    moda = serie.mode()

    if not moda.empty:
        return moda.iloc[0]

    return serie.iloc[0]

## 02. Regras de combustível

In [36]:

# ==========================
# 1. Taxas por CODACT e mês
# ==========================
taxas_ingresso = {
    11: {
        3: 0.0637, 4: 0.1043, 5: 0.0869,
        6: 0.0569, 7: 0.0665, 8: 0.1000,
        9: 0.1229,
    },

    74: {
        3: 0.0646, 4: 0.1003, 5: 0.0850,
        6: 0.0585, 7: 0.0670, 8: 0.0965,
        9: 0.1167,
    },

    84: {
        3: 0.0646, 4: 0.1003, 5: 0.0850,
        6: 0.0585, 7: 0.0670, 8: 0.0965,
        9: 0.1167,
    },

    118: {
        1: 0.0849, 2: 0.0974, 3: 0.1860,
        4: 0.2434, 5: 0.2188, 6: 0.1763,
        7: 0.1900, 8: 0.2373, 9: 0.2698,
    },

    131: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    262: {
        3: 0.0646, 4: 0.1003, 5: 0.0850,
        6: 0.0585, 7: 0.0670, 8: 0.0965,
        9: 0.1167,
    },

    275: {
        3: 0.0669, 4: 0.1077, 5: 0.0902,
        6: 0.0600, 7: 0.0697, 8: 0.1034,
        9: 0.1265,
    },

    280: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    282: {
        3: 0.0646, 4: 0.1003, 5: 0.0850,
        6: 0.0585, 7: 0.0670, 8: 0.0965,
        9: 0.1167,
    },

    332: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    391: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    398: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    403: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    404: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    405: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    415: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    418: {
        9: 0.1361,
    },

    419: {
        1: 0.0442, 2: 0.0531, 3: 0.1160,
        4: 0.1568, 5: 0.1393, 6: 0.1091,
        7: 0.1188, 8: 0.1524, 9: 0.1755,
    },

    429: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    588: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    660: {
        3: 0.0480, 4: 0.0821, 5: 0.0675,
        6: 0.0422, 7: 0.0504, 8: 0.0785,
        9: 0.0978,
    },

    680: {
        3: 0.0646, 4: 0.1003, 5: 0.0850,
        6: 0.0585, 7: 0.0670, 8: 0.0965,
        9: 0.1167,
    },

    784: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },
}


## 03. Importações

In [37]:
# ==========================
# 1. Validar fontes
# ==========================
if not PATHS["db"].exists():
    raise FileNotFoundError(
        f"DuckDB não encontrada: {PATHS['db']}"
    )

if not PATHS["excel"].exists():
    raise FileNotFoundError(
        f"Excel Danone não encontrado: {PATHS['excel']}"
    )

# ==========================
# 2. Ler DuckDB
# ==========================
with duckdb.connect(
    str(PATHS["db"]),
    read_only=True,
) as con:
    df_raw = con.sql("""
        SELECT *
        FROM inform_27_2026
    """).df()

    df_tarifas_raw = con.sql("""
        SELECT
            tipo_local,
            modelo_ingresso,
            tarifa_base,
            data_inicio,
            data_fim
        FROM Danone_Tarifas_2026
    """).df()

    coords_raw = con.sql("""
        SELECT
            cp AS CP,
            point_x AS POINT_X,
            point_y AS POINT_Y
        FROM coordenadas
    """).df()

# ==========================
# 3. Ler KILOSPO
# ==========================
raw_kilospo = pd.read_excel(
    PATHS["excel"],
    sheet_name="KILOSPO",
    header=None,
)

df_kilospo_raw = raw_kilospo.iloc[1:].copy()
df_kilospo_raw.columns = raw_kilospo.iloc[0]

# ==========================
# 4. Validar importações
# ==========================
pd.DataFrame({
    "fonte": [
        "inform_27_2026",
        "KILOSPO",
        "Danone_Tarifas_2026",
        "coordenadas",
    ],
    "linhas": [
        len(df_raw),
        len(df_kilospo_raw),
        len(df_tarifas_raw),
        len(coords_raw),
    ],
})

,fonte,linhas
0,inform_27_2026,632022
1,KILOSPO,38973
2,Danone_Tarifas_2026,37
3,coordenadas,178409


## 04. Universo financeiro — `inform_27_2026`

In [38]:
# ==========================
# 1. Cópia de trabalho
# ==========================
df_base = df_raw.copy()

# ==========================
# 2. Texto
# ==========================
for coluna in df_base.select_dtypes(
    include=["object"]
).columns:
    df_base[coluna] = (
        df_base[coluna]
        .astype("string")
        .str.strip()
    )

# ==========================
# 3. Numéricos
# ==========================
colunas_numericas = [
    "CODACT",
    "INGRESODT",
    "COSTEDT",
    "RENTADT",
    "PALETSDT",
    "PESO_BRUTO",
    "PALETS",
    "KM",
    "KMREALES",
    "CAMION_CAPACIDAD",
]

for coluna in colunas_numericas:
    if coluna in df_base.columns:
        df_base[coluna] = pd.to_numeric(
            df_base[coluna],
            errors="coerce",
        )

# ==========================
# 4. CODEUT
# ==========================
df_base["CODEUT"] = (
    df_base["CODEUT"]
    .astype("string")
    .str.strip()
    .replace({
        "": pd.NA,
        "nan": pd.NA,
        "None": pd.NA,
        "<NA>": pd.NA,
    })
)

# ==========================
# 5. Datas
# ==========================
df_base["FCARGA"] = normalizar_data(
    df_base["FCARGA"]
)

df_base["FENTREGA"] = normalizar_data(
    df_base["FENTREGA"]
)

# ==========================
# 6. Chave de referência
# ==========================
df_base["referencia_chave"] = (
    normalizar_referencia(
        df_base["REFERENCIA"]
    )
)

# ==========================
# 7. Guardar universo antes dos filtros
# ==========================
df_raw_check = df_base.copy()

# ==========================
# 8. Ajustes de ingresso
# ==========================
df_base["INGRESODT_ORIGINAL"] = (
    df_base["INGRESODT"]
)

df_base.loc[
    df_base["CODACT"].eq(11),
    "INGRESODT",
] = 0.0

mask_espanha = df_base["CODACT"].isin(
    CODACT_ESPANHA
)

df_base.loc[
    mask_espanha,
    "INGRESODT",
] = (
    df_base.loc[
        mask_espanha,
        "COSTEDT",
    ]
    * 1.05
)

# ==========================
# 9. Excluir CODACT 13
# ==========================
df_base = df_base.loc[
    ~df_base["CODACT"].isin(
        CODACT_EXCLUIR
    )
].copy()

# ==========================
# 10. Filtro operacional
# ==========================
mask_gestao = (
    df_base["GESTION"].eq("LIS")
    | (
        df_base["GESTION"].isna()
        & df_base["PROPIETARIO"].isin(
            ["CEP", "PTG"]
        )
    )
)

df_base = df_base.loc[
    mask_gestao
].copy()

# ==========================
# 11. ID e calendário
# ==========================
df_base["_linha_id"] = np.arange(
    len(df_base),
    dtype="int64",
)

df_base["data"] = df_base["FENTREGA"]
df_base["ano"] = df_base["data"].dt.year
df_base["mes"] = df_base["data"].dt.month
df_base["mes_nome"] = df_base["data"].dt.month_name()
df_base["week_number"] = (
    df_base["data"]
    .dt.isocalendar()
    .week
    .astype("Int64")
)
df_base["week_day"] = (
    df_base["data"]
    .dt.day_name()
)
df_base["mes_estrutura"] = (
    df_base["data"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

# ==========================
# 12. Validação
# ==========================
pd.DataFrame({
    "metrica": [
        "Linhas raw",
        "Linhas universo financeiro",
        "CODACT 13 restantes",
        "CODEUT nulos",
        "FENTREGA nula",
    ],
    "valor": [
        len(df_raw),
        len(df_base),
        df_base["CODACT"].eq(13).sum(),
        df_base["CODEUT"].isna().sum(),
        df_base["FENTREGA"].isna().sum(),
    ],
})


C:\Users\LISARR\AppData\Local\Temp\ipykernel_21620\385853508.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for coluna in df_base.select_dtypes(


,metrica,valor
0,Linhas raw,632022
1,Linhas universo financeiro,98239
2,CODACT 13 restantes,0
3,CODEUT nulos,0
4,FENTREGA nula,0


## 05. Danone — cálculo operacional

A KILOSPO é calculada integralmente.  
O ingresso financeiro só é reconhecido quando existe associação ao universo financeiro.

A diferença entre KILOSPO e `inform_27` é classificada, não forçada.

In [39]:
# ==========================
# 1. Preparar KILOSPO
# ==========================
df_danone = df_kilospo_raw.copy()

df_danone["referencia_chave"] = (
    normalizar_referencia(
        df_danone["PREFPE"]
    )
)

df_danone["PFEENT"] = normalizar_data(
    df_danone["PFEENT"]
)

df_danone["PCODCL"] = (
    df_danone["PCODCL"]
    .astype("string")
    .str.strip()
)

df_danone["tipo_local"] = (
    df_danone["Ruta Tte. Nueva"]
    .astype("string")
    .str.strip()
)

df_danone["Kg Neto Entregado"] = (
    pd.to_numeric(
        df_danone["Kg Neto Entregado"],
        errors="coerce",
    )
)

df_danone = df_danone.loc[
    df_danone["PFEENT"].dt.year.eq(
        ANO
    )
].copy()

# ==========================
# 2. Preparar tarifas
# ==========================
df_tarifas = df_tarifas_raw.copy()

df_tarifas["tipo_local"] = (
    df_tarifas["tipo_local"]
    .astype("string")
    .str.strip()
)

df_tarifas["modelo_ingresso"] = (
    df_tarifas["modelo_ingresso"]
    .astype("string")
    .str.strip()
)

df_tarifas["tarifa_base"] = (
    pd.to_numeric(
        df_tarifas["tarifa_base"],
        errors="coerce",
    )
)

df_tarifas["data_inicio"] = (
    pd.to_datetime(
        df_tarifas["data_inicio"],
        errors="coerce",
    )
)

df_tarifas["data_fim"] = (
    pd.to_datetime(
        df_tarifas["data_fim"],
        errors="coerce",
    )
)

if (
    df_tarifas["tipo_local"].isna().any()
    or df_tarifas["data_inicio"].isna().any()
):
    raise ValueError(
        "Existem tarifas sem tipo_local ou data_inicio."
    )

mask_intervalo_invalido = (
    df_tarifas["data_fim"].notna()
    & df_tarifas["data_fim"].lt(
        df_tarifas["data_inicio"]
    )
)

if mask_intervalo_invalido.any():
    raise ValueError(
        "Existem tarifas com data_fim anterior a data_inicio."
    )

# ==========================
# 3. Validar sobreposição
# ==========================
tarifas_ord = (
    df_tarifas
    .sort_values(
        ["tipo_local", "data_inicio"]
    )
    .copy()
)

tarifas_ord["fim_validacao"] = (
    tarifas_ord["data_fim"]
    .fillna(pd.Timestamp.max.normalize())
)

tarifas_ord["fim_anterior"] = (
    tarifas_ord
    .groupby("tipo_local")[
        "fim_validacao"
    ]
    .shift(1)
)

mask_sobreposicao = (
    tarifas_ord["fim_anterior"].notna()
    & tarifas_ord["data_inicio"].le(
        tarifas_ord["fim_anterior"]
    )
)

if mask_sobreposicao.any():
    tipos = (
        tarifas_ord.loc[
            mask_sobreposicao,
            "tipo_local",
        ]
        .dropna()
        .unique()
        .tolist()
    )
    raise ValueError(
        "Existem períodos tarifários sobrepostos: "
        f"{tipos}"
    )

# ==========================
# 4. Associar tarifa por data
# ==========================
df_danone["modelo_ingresso"] = pd.Series(
    pd.NA,
    index=df_danone.index,
    dtype="string",
)

df_danone["tarifa_base"] = pd.Series(
    pd.NA,
    index=df_danone.index,
    dtype="Float64",
)

df_danone["tarifa_data_inicio"] = pd.NaT
df_danone["tarifa_data_fim"] = pd.NaT

tarifa_atribuida = pd.Series(
    False,
    index=df_danone.index,
    dtype="bool",
)

for tarifa in df_tarifas.itertuples(index=False):
    mask_tarifa = (
        df_danone["tipo_local"].eq(
            tarifa.tipo_local
        )
        & df_danone["PFEENT"].ge(
            tarifa.data_inicio
        )
    ).fillna(False)

    if pd.notna(tarifa.data_fim):
        mask_tarifa &= (
            df_danone["PFEENT"].le(
                tarifa.data_fim
            )
            .fillna(False)
        )

    if (tarifa_atribuida & mask_tarifa).any():
        raise ValueError(
            "Mais de uma tarifa aplicável à mesma entrega Danone."
        )

    df_danone.loc[
        mask_tarifa,
        "modelo_ingresso",
    ] = tarifa.modelo_ingresso

    df_danone.loc[
        mask_tarifa,
        "tarifa_base",
    ] = tarifa.tarifa_base

    df_danone.loc[
        mask_tarifa,
        "tarifa_data_inicio",
    ] = tarifa.data_inicio

    df_danone.loc[
        mask_tarifa,
        "tarifa_data_fim",
    ] = tarifa.data_fim

    tarifa_atribuida |= mask_tarifa

# ==========================
# 5. Ingresso base
# ==========================
df_danone["ingresso_danone_base"] = (
    pd.Series(
        pd.NA,
        index=df_danone.index,
        dtype="Float64",
    )
)

mask_ton = (
    df_danone["modelo_ingresso"]
    .eq("POR_TONELADA")
    .fillna(False)
)

mask_entrega = (
    df_danone["modelo_ingresso"]
    .eq("POR_ENTREGA")
    .fillna(False)
)

mask_sem = (
    df_danone["modelo_ingresso"]
    .eq("SEM_INGRESSO")
    .fillna(False)
)

df_danone.loc[
    mask_ton,
    "ingresso_danone_base",
] = (
    df_danone.loc[
        mask_ton,
        "Kg Neto Entregado",
    ]
    / 1000
    * df_danone.loc[
        mask_ton,
        "tarifa_base",
    ]
)

df_danone.loc[
    mask_entrega,
    "ingresso_danone_base",
] = (
    df_danone.loc[
        mask_entrega,
        "tarifa_base",
    ]
)

df_danone.loc[
    mask_sem,
    "ingresso_danone_base",
] = 0.0

# ==========================
# 6. Extra mensal
# ==========================
df_danone["mes_periodo"] = (
    df_danone["PFEENT"]
    .dt.to_period("M")
)

df_danone["extra_danone"] = 0.0

mask_extra = (
    df_danone["ingresso_danone_base"]
    .notna()
    & df_danone["ingresso_danone_base"]
    .gt(0)
    & df_danone["mes_periodo"]
    .notna()
)

total_base_mes = (
    df_danone["ingresso_danone_base"]
    .where(mask_extra)
    .groupby(
        df_danone["mes_periodo"]
    )
    .transform("sum")
)

if EXTRA_DANONE_MENSAL != 0:
    df_danone.loc[
        mask_extra,
        "extra_danone",
    ] = (
        df_danone.loc[
            mask_extra,
            "ingresso_danone_base",
        ]
        .div(
            total_base_mes.loc[
                mask_extra
            ]
        )
        .mul(EXTRA_DANONE_MENSAL)
    )

df_danone["ingresso_danone"] = (
    df_danone["ingresso_danone_base"]
    + df_danone["extra_danone"]
)

# ==========================
# 7. Estado do cálculo
# ==========================
condicoes = [
    (
        df_danone["modelo_ingresso"]
        .eq("POR_TONELADA")
        & df_danone["ingresso_danone"]
        .notna()
    ).fillna(False).to_numpy(dtype=bool),

    (
        df_danone["modelo_ingresso"]
        .eq("POR_ENTREGA")
        & df_danone["ingresso_danone"]
        .notna()
    ).fillna(False).to_numpy(dtype=bool),

    df_danone["modelo_ingresso"]
    .eq("SEM_INGRESSO")
    .fillna(False)
    .to_numpy(dtype=bool),

    df_danone["modelo_ingresso"]
    .eq("CAPILAR")
    .fillna(False)
    .to_numpy(dtype=bool),

    df_danone["modelo_ingresso"]
    .eq("POR_DEFINIR")
    .fillna(False)
    .to_numpy(dtype=bool),

    df_danone["modelo_ingresso"]
    .isna()
    .to_numpy(dtype=bool),
]

escolhas = [
    "CALCULADO_POR_TONELADA",
    "CALCULADO_POR_ENTREGA",
    "SEM_INGRESSO",
    "CAPILAR",
    "POR_DEFINIR",
    "SEM_TARIFA",
]

df_danone["estado_calculo"] = (
    np.select(
        condicoes,
        escolhas,
        default="REVER",
    )
)

# ==========================
# 8. Validação mensal
# ==========================
validacao_danone_mes = (
    df_danone
    .groupby(
        "mes_periodo",
        as_index=False,
    )
    .agg(
        linhas=(
            "referencia_chave",
            "size",
        ),
        ingresso_base=(
            "ingresso_danone_base",
            lambda x: x.sum(
                min_count=1
            ),
        ),
        extra=(
            "extra_danone",
            "sum",
        ),
        ingresso_final=(
            "ingresso_danone",
            lambda x: x.sum(
                min_count=1
            ),
        ),
    )
)

validacao_danone_mes


,mes_periodo,linhas,ingresso_base,extra,ingresso_final
0,2026-09,5398,143968.553183,20000.0,163968.553183


In [40]:
df_danone

,BLIINF,CODACT,PREFPE,PFEENT,PCODCL,PNOMCL,PRUTA,PCATCL,CODTLI,CODMOP,...,tipo_local,modelo_ingresso,tarifa_base,tarifa_data_inicio,tarifa_data_fim,ingresso_danone_base,mes_periodo,extra_danone,ingresso_danone,estado_calculo
1,FGE50PTG,011,5023683578 415802375,2026-09-01,350331388,"PINGO DOCE - DIST. ALIMENTAR,",2000,DI2,STD,PSG,...,Azambuja 3,POR_TONELADA,52.488448,2026-01-01,NaT,0.0,2026-09,0.000000,0.0,CALCULADO_POR_TONELADA
2,FGE50PTG,011,5023656430 415821120,2026-09-01,350148119,PINGO DOCE AZAMBUJA DIST. ALIM,2000,DI2,STD,PD5,...,Azambuja 1,POR_TONELADA,14.434323,2026-01-01,NaT,130.327907,2026-09,18.105052,148.432958,CALCULADO_POR_TONELADA
3,FGE50PTG,011,5023687927 415821120,2026-09-01,350148119,PINGO DOCE AZAMBUJA DIST. ALIM,2000,DI2,STD,PD5,...,Azambuja 1,POR_TONELADA,14.434323,2026-01-01,NaT,0.0,2026-09,0.000000,0.0,CALCULADO_POR_TONELADA
4,FGE50PTG,011,5023708921 415821121,2026-09-01,350152530,MODELO CONT.HIP.SA-MAIA,2600,DI2,STD,IT1,...,Norte 1,POR_TONELADA,31.493069,2026-01-01,NaT,1.870688,2026-09,0.259875,2.130563,CALCULADO_POR_TONELADA
5,FGE50PTG,011,5023695448 415821128,2026-09-01,350152530,MODELO CONT.HIP.SA-MAIA,2600,DI2,STD,IT1,...,Norte 1,POR_TONELADA,31.493069,2026-01-01,NaT,397.486621,2026-09,55.218534,452.705155,CALCULADO_POR_TONELADA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5394,FGE50PTG,011,5023952566 416055501,2026-09-30,350464011,OFERTAS ENTREGA UNICA,9700,OBS,STD,COM,...,<NA>,<NA>,<NA>,NaT,NaT,<NA>,2026-09,0.000000,<NA>,SEM_TARIFA
5395,FGE50PTG,011,5023881624 416041691,2026-09-30,350421219,El CORTE INGLES PLATAF. ALCOCH,706,DI2,STD,EC8,...,Norte 3,POR_TONELADA,78.732672,2026-01-01,NaT,181.717684,2026-09,25.244080,206.961764,CALCULADO_POR_TONELADA
5396,FGE50PTG,011,5023920928 416020745,2026-09-30,350460342,FERNANDO M. PACHECO - REFRIGER,8800,AC1,STD,PDS,...,Azambuja 1,POR_TONELADA,14.434323,2026-01-01,NaT,2.483281,2026-09,0.344975,2.828256,CALCULADO_POR_TONELADA
5397,FGE50PTG,011,5023920188 416020746,2026-09-30,350460342,FERNANDO M. PACHECO - REFRIGER,8800,AC1,STD,PDS,...,Azambuja 1,POR_TONELADA,14.434323,2026-01-01,NaT,12.168134,2026-09,1.690388,13.858522,CALCULADO_POR_TONELADA


## 06. Associação Danone e reconciliação

In [41]:
# ==========================
# 1. Agregar por referência/data
# ==========================

danone_chave = (
    df_danone.loc[
        df_danone["referencia_chave"].notna()
        & df_danone["PFEENT"].notna()
    ]
    .groupby(
        [
            "referencia_chave",
            "PFEENT",
        ],
        as_index=False,
    )
    .agg(
        ingresso_danone_total=(
            "ingresso_danone",
            lambda x: x.sum(
                min_count=1
            ),
        ),
        ingresso_danone_base_total=(
            "ingresso_danone_base",
            lambda x: x.sum(
                min_count=1
            ),
        ),
        extra_danone_total=(
            "extra_danone",
            "sum",
        ),
    )
)


# ==========================
# 2. Associar exacto
# ==========================

df = df_base.copy()

df = df.merge(
    danone_chave,
    left_on=[
        "referencia_chave",
        "FENTREGA",
    ],
    right_on=[
        "referencia_chave",
        "PFEENT",
    ],
    how="left",
    validate="many_to_one",
)


# ==========================
# 3. Identificar match
# ==========================

tem_match = (
    df["ingresso_danone_total"]
    .notna()
)


# ==========================
# 4. Identificar destino final
# ==========================

destino_preenchido = (
    df["DESTINO"]
    .astype("string")
    .fillna("")
    .str.strip()
    .ne("")
)

n_destinos = (
    destino_preenchido
    .groupby(
        [
            df["referencia_chave"],
            df["FENTREGA"],
        ],
        dropna=False,
    )
    .transform("sum")
)


# ==========================
# 5. Escolher linha Danone
# ==========================

linha_destino_unico = (
    tem_match
    & destino_preenchido
    & n_destinos.eq(1)
)

linha_fallback = (
    tem_match
    & ~n_destinos.eq(1)
    & ~df.duplicated(
        subset=[
            "referencia_chave",
            "FENTREGA",
        ],
        keep="last",
    )
)

linha_ingresso_danone = (
    linha_destino_unico
    | linha_fallback
)


# ==========================
# 6. Atribuir ingresso Danone
# ==========================

df["ingresso_danone"] = (
    pd.Series(
        pd.NA,
        index=df.index,
        dtype="Float64",
    )
)

df.loc[
    linha_ingresso_danone,
    "ingresso_danone",
] = df.loc[
    linha_ingresso_danone,
    "ingresso_danone_total",
]


# ==========================
# 7. Limpar colunas auxiliares
# ==========================

df = df.drop(
    columns=[
        "PFEENT",
        "ingresso_danone_total",
        "ingresso_danone_base_total",
        "extra_danone_total",
    ],
    errors="ignore",
)


# ==========================
# 8. Ingresso total
# ==========================

df["total_ingresso"] = (
    df["INGRESODT"].fillna(0.0)
    + df["ingresso_danone"].fillna(0.0)
)


In [42]:
# ==========================
# 1. Âmbito de validação
# ==========================
danone_validacao = (
    danone_chave.loc[
        danone_chave["PFEENT"]
        .dt.month
        .le(MES_VALIDACAO_DANONE_MAX)
    ]
    .copy()
)

base_validacao = (
    df_base.loc[
        df_base["FENTREGA"]
        .dt.month
        .le(MES_VALIDACAO_DANONE_MAX)
    ]
    .copy()
)

raw_validacao = (
    df_raw_check.loc[
        df_raw_check["FENTREGA"]
        .dt.month
        .le(MES_VALIDACAO_DANONE_MAX)
        | df_raw_check["FENTREGA"].isna()
    ]
    .copy()
)

# ==========================
# 2. Chaves disponíveis
# ==========================
chaves_base_exactas = set(
    zip(
        base_validacao["referencia_chave"],
        base_validacao["FENTREGA"],
    )
)

refs_base = set(
    base_validacao[
        "referencia_chave"
    ].dropna()
)

refs_raw = set(
    raw_validacao[
        "referencia_chave"
    ].dropna()
)

refs_codact13 = set(
    raw_validacao.loc[
        raw_validacao["CODACT"].eq(13),
        "referencia_chave",
    ].dropna()
)

# ==========================
# 3. Classificar diferença
# ==========================
def classificar_danone(row):
    chave = (
        row["referencia_chave"],
        row["PFEENT"],
    )

    ref = row["referencia_chave"]

    if chave in chaves_base_exactas:
        return "ASSOCIADO_EXACTO"

    if ref in refs_codact13:
        return "EXCLUIDO_CODACT_13"

    if ref in refs_base:
        return "REFERENCIA_EXISTE_DATA_DIFERENTE"

    if ref in refs_raw:
        return "EXCLUIDO_OUTRO_FILTRO"

    return "FORA_INFORM27"

danone_validacao[
    "estado_reconciliacao"
] = danone_validacao.apply(
    classificar_danone,
    axis=1,
)

# ==========================
# 4. Resumo
# ==========================
validacao_danone = (
    danone_validacao
    .groupby(
        "estado_reconciliacao",
        as_index=False,
    )
    .agg(
        casos=(
            "referencia_chave",
            "size",
        ),
        referencias=(
            "referencia_chave",
            "nunique",
        ),
        ingresso_base=(
            "ingresso_danone_base_total",
            "sum",
        ),
        extra=(
            "extra_danone_total",
            "sum",
        ),
        ingresso=(
            "ingresso_danone_total",
            "sum",
        ),
    )
    .sort_values(
        "ingresso",
        ascending=False,
    )
    .reset_index(drop=True)
)

# ==========================
# 5. Reconciliação financeira real
# ==========================
df_validacao_danone = (
    df.loc[
        df["FENTREGA"]
        .dt.month
        .le(MES_VALIDACAO_DANONE_MAX)
    ]
    .copy()
)

total_danone_df = (
    df_validacao_danone[
        "ingresso_danone"
    ]
    .fillna(0.0)
    .sum()
)

total_danone_associado_exacto = (
    danone_validacao.loc[
        danone_validacao[
            "estado_reconciliacao"
        ].eq("ASSOCIADO_EXACTO"),
        "ingresso_danone_total",
    ]
    .fillna(0.0)
    .sum()
)

erro_reconciliacao_danone = (
    total_danone_df
    - total_danone_associado_exacto
)

# ==========================
# 6. Validar troço final
# ==========================
validacao_troco_danone = (
    df.loc[
        df["referencia_chave"].notna()
        & df["FENTREGA"].notna()
    ]
    .copy()
)

validacao_troco_danone[
    "destino_preenchido"
] = (
    validacao_troco_danone["DESTINO"]
    .astype("string")
    .fillna("")
    .str.strip()
    .ne("")
)

validacao_troco_danone[
    "n_destinos_chave"
] = (
    validacao_troco_danone
    .groupby(
        ["referencia_chave", "FENTREGA"]
    )["destino_preenchido"]
    .transform("sum")
)

casos_troco_danone_errados = int(
    (
        validacao_troco_danone[
            "ingresso_danone"
        ].notna()
        & validacao_troco_danone[
            "n_destinos_chave"
        ].eq(1)
        & ~validacao_troco_danone[
            "destino_preenchido"
        ]
    ).sum()
)

validacao_danone


,estado_reconciliacao,casos,referencias,ingresso_base,extra,ingresso
0,ASSOCIADO_EXACTO,1678,1678,137278.934637,19070.683368,156349.618005
1,EXCLUIDO_CODACT_13,3154,3154,5755.858858,799.599458,6555.458316
2,FORA_INFORM27,263,263,781.260631,108.532122,889.792754
3,REFERENCIA_EXISTE_DATA_DIFERENTE,6,6,152.499057,21.185051,173.684108


## 07. Combustível

In [43]:
# ==========================
# 1. Mês da taxa
# ==========================
df["mes_taxa_combustivel"] = (
    df["FENTREGA"].dt.month
    + DESFASAMENTO_MES_COMBUSTIVEL
)

# ==========================
# 2. Taxa
# ==========================
def obter_taxa(codact, mes):
    if pd.isna(codact) or pd.isna(mes):
        return np.nan

    mes = int(mes)

    if mes < 1 or mes > 12:
        return np.nan

    return taxas_ingresso.get(
        int(codact),
        {},
    ).get(
        mes,
        np.nan,
    )


def taxa_configurada_no_periodo(codact, mes):
    if pd.isna(codact) or pd.isna(mes):
        return False

    taxas_codact = taxas_ingresso.get(
        int(codact),
        {},
    )

    if not taxas_codact:
        return False

    mes = int(mes)
    meses = sorted(taxas_codact)

    return (
        min(meses)
        <= mes
        <= max(meses)
    )


df["taxa_combustivel"] = [
    obter_taxa(codact, mes)
    for codact, mes in zip(
        df["CODACT"],
        df["mes_taxa_combustivel"],
    )
]

df["taxa_combustivel"] = (
    pd.to_numeric(
        df["taxa_combustivel"],
        errors="coerce",
    )
)

# ==========================
# 3. Base combustível
# ==========================
df["base_combustivel"] = (
    pd.to_numeric(
        df["INGRESODT"],
        errors="coerce",
    )
)

mask_codact11 = (
    df["CODACT"].eq(11)
)

df.loc[
    mask_codact11,
    "base_combustivel",
] = (
    pd.to_numeric(
        df.loc[
            mask_codact11,
            "ingresso_danone",
        ],
        errors="coerce",
    )
)

# ==========================
# 4. Validar taxa em falta
# ==========================
df["taxa_combustivel_esperada"] = [
    taxa_configurada_no_periodo(
        codact,
        mes,
    )
    for codact, mes in zip(
        df["CODACT"],
        df["mes_taxa_combustivel"],
    )
]

df["taxa_combustivel_em_falta"] = (
    df["taxa_combustivel_esperada"]
    & df["taxa_combustivel"].isna()
    & df["base_combustivel"]
    .fillna(0.0)
    .ne(0.0)
)

# ==========================
# 5. Acerto combustível
# ==========================
df["acerto_combustivel"] = 0.0

mask_taxa = (
    df["taxa_combustivel"].notna()
    & df["base_combustivel"].notna()
)

df.loc[
    mask_taxa,
    "acerto_combustivel",
] = (
    df.loc[
        mask_taxa,
        "base_combustivel",
    ]
    * df.loc[
        mask_taxa,
        "taxa_combustivel",
    ]
)

df["total_ingresso_c"] = (
    df["total_ingresso"]
    + df["acerto_combustivel"]
)


## 08. Capacidade e coordenadas

In [44]:
# ==========================
# 1. Capacidade
# ==========================

df["CAMION_CAPACIDAD_NUM"] = (
    pd.to_numeric(
        df["CAMION_CAPACIDAD"],
        errors="coerce",
    )
)

df["capacidade_norm"] = (
    df["CAMION_CAPACIDAD_NUM"]
    .map(MAPA_CAPACIDADE)
)

capacidade_codeut = (
    df.loc[
        df["CODEUT"].notna()
        & df["capacidade_norm"].notna()
    ]
    .groupby("CODEUT")[
        "capacidade_norm"
    ]
    .agg(moda_segura)
)

mask_sem_capacidade = (
    df["CODEUT"].notna()
    & df["capacidade_norm"].isna()
)

df.loc[
    mask_sem_capacidade,
    "capacidade_norm",
] = (
    df.loc[
        mask_sem_capacidade,
        "CODEUT",
    ]
    .map(capacidade_codeut)
)


# ==========================
# 2. Paletes por CODEUT
# ==========================

paletes_codeut = (
    df.loc[
        df["CODEUT"].notna()
    ]
    .groupby("CODEUT")[
        "PALETS"
    ]
    .sum(min_count=1)
)


# ==========================
# 3. Corrigir capacidade 33
# ==========================

df["paletes_codeut"] = (
    df["CODEUT"]
    .map(paletes_codeut)
)

mask_capacidade_33 = (
    df["capacidade_norm"].eq(33)
)

df.loc[
    mask_capacidade_33
    & df["paletes_codeut"].gt(40)
    & df["paletes_codeut"].le(55),
    "capacidade_norm",
] = 55

df.loc[
    mask_capacidade_33
    & df["paletes_codeut"].gt(55),
    "capacidade_norm",
] = 66


# ==========================
# 4. Paletes para ocupação
# ==========================

df["paletes_ocupacao_codeut"] = (
    df["paletes_codeut"]
    .clip(upper=66)
)


# ==========================
# 5. Dados válidos
# ==========================

df["dados_validos"] = (
    df["CODEUT"].notna()
    & df["FENTREGA"].notna()
    & df["capacidade_norm"].notna()
    & df["paletes_ocupacao_codeut"].notna()
)


In [45]:
# ==========================
# 1. Preparar coordenadas
# ==========================
coords = coords_raw.copy()

coords["POINT_X"] = pd.to_numeric(
    coords["POINT_X"]
    .astype("string")
    .str.strip()
    .str.replace(",", ".", regex=False),
    errors="coerce",
)

coords["POINT_Y"] = pd.to_numeric(
    coords["POINT_Y"]
    .astype("string")
    .str.strip()
    .str.replace(",", ".", regex=False),
    errors="coerce",
)

coords["CP_chave"] = normalizar_cp(
    coords["CP"]
)

coords["CP_parte"] = (
    coords["CP_chave"]
    .str[:4]
)

df["CPOSTAL_chave"] = normalizar_cp(
    df["CPOSTAL"]
)

df["CPOSTAD_chave"] = normalizar_cp(
    df["CPOSTAD"]
)

df["CPOSTAL_parte"] = (
    df["CPOSTAL_chave"]
    .str[:4]
)

df["CPOSTAD_parte"] = (
    df["CPOSTAD_chave"]
    .str[:4]
)

# ==========================
# 2. CP exacto
# ==========================
coords_cp = (
    coords.dropna(
        subset=[
            "CP_chave",
            "POINT_X",
            "POINT_Y",
        ]
    )
    .groupby(
        "CP_chave",
        as_index=False,
    )[
        ["POINT_X", "POINT_Y"]
    ]
    .mean()
)

origem = coords_cp.rename(
    columns={
        "CP_chave": "CPOSTAL_chave",
        "POINT_X": "longitude_origem",
        "POINT_Y": "latitude_origem",
    }
)

destino = coords_cp.rename(
    columns={
        "CP_chave": "CPOSTAD_chave",
        "POINT_X": "longitude_destino",
        "POINT_Y": "latitude_destino",
    }
)

df = df.merge(
    origem,
    on="CPOSTAL_chave",
    how="left",
    validate="many_to_one",
)

df = df.merge(
    destino,
    on="CPOSTAD_chave",
    how="left",
    validate="many_to_one",
)

# ==========================
# 3. Fallback 4 dígitos
# ==========================
centroid = (
    coords.dropna(
        subset=[
            "CP_parte",
            "POINT_X",
            "POINT_Y",
        ]
    )
    .groupby(
        "CP_parte",
        as_index=False,
    )[
        ["POINT_X", "POINT_Y"]
    ]
    .mean()
)

centroid_origem = centroid.rename(
    columns={
        "CP_parte": "CPOSTAL_parte",
        "POINT_X": "longitude_origem_fb",
        "POINT_Y": "latitude_origem_fb",
    }
)

centroid_destino = centroid.rename(
    columns={
        "CP_parte": "CPOSTAD_parte",
        "POINT_X": "longitude_destino_fb",
        "POINT_Y": "latitude_destino_fb",
    }
)

df = df.merge(
    centroid_origem,
    on="CPOSTAL_parte",
    how="left",
    validate="many_to_one",
)

df = df.merge(
    centroid_destino,
    on="CPOSTAD_parte",
    how="left",
    validate="many_to_one",
)

df["longitude_origem"] = (
    df["longitude_origem"]
    .fillna(
        df["longitude_origem_fb"]
    )
)

df["latitude_origem"] = (
    df["latitude_origem"]
    .fillna(
        df["latitude_origem_fb"]
    )
)

df["longitude_destino"] = (
    df["longitude_destino"]
    .fillna(
        df["longitude_destino_fb"]
    )
)

df["latitude_destino"] = (
    df["latitude_destino"]
    .fillna(
        df["latitude_destino_fb"]
    )
)

df = df.drop(
    columns=[
        "longitude_origem_fb",
        "latitude_origem_fb",
        "longitude_destino_fb",
        "latitude_destino_fb",
    ],
    errors="ignore",
)

## 09. Custos, viagens e rentabilidade

In [46]:
# ==========================
# 1. Preparar valores
# ==========================
df["PALETS"] = pd.to_numeric(
    df["PALETS"],
    errors="coerce",
).fillna(0.0)

df["COSTEDT"] = pd.to_numeric(
    df["COSTEDT"],
    errors="coerce",
).fillna(0.0)

# ==========================
# 2. Paletes viagem/mês
# ==========================
paletes_codeut_mes = (
    df.groupby(
        [
            "mes_estrutura",
            "CODEUT",
        ],
        dropna=False,
    )["PALETS"]
    .transform("sum")
)

mask_estrutura = (
    df["mes_estrutura"].notna()
    & df["CODEUT"].notna()
    & paletes_codeut_mes.gt(0)
)

# ==========================
# 3. Viagens mensais
# ==========================
n_codeut_mes = (
    df.loc[
        mask_estrutura
    ]
    .groupby(
        "mes_estrutura"
    )["CODEUT"]
    .nunique()
)

df["n_viagens_estrutura_mes"] = (
    df["mes_estrutura"]
    .map(n_codeut_mes)
)

df["custo_estrutura_viagem"] = (
    CUSTO_ESTRUTURA_MENSAL
    / df["n_viagens_estrutura_mes"]
)

# ==========================
# 4. Rateio estrutura
# ==========================
df["custo_estrutura"] = 0.0

df.loc[
    mask_estrutura,
    "custo_estrutura",
] = (
    df.loc[
        mask_estrutura,
        "custo_estrutura_viagem",
    ]
    * df.loc[
        mask_estrutura,
        "PALETS",
    ]
    / paletes_codeut_mes.loc[
        mask_estrutura
    ]
)

# ==========================
# 5. Custo e margem
# ==========================
df["custo_total_c"] = (
    df["COSTEDT"]
    + df["custo_estrutura"]
)

df["margem"] = (
    df["total_ingresso_c"]
    - df["custo_total_c"]
)

df["custo_por_palete"] = (
    dividir_seguro(
        df["custo_total_c"],
        df["PALETS"],
    )
)

df["ingresso_por_palete"] = (
    dividir_seguro(
        df["total_ingresso_c"],
        df["PALETS"],
    )
)

df["margem_por_palete"] = (
    dividir_seguro(
        df["margem"],
        df["PALETS"],
    )
)

df["margem_pct"] = (
    dividir_seguro(
        df["margem"],
        df["total_ingresso_c"],
    )
    * 100
)

In [47]:
# ==========================
# 1. Tabela oficial de viagens
# ==========================
df_viagens = (
    df.loc[
        df["CODEUT"].notna()
    ]
    .groupby(
        "CODEUT",
        as_index=False,
    )
    .agg(
        data=("FENTREGA", "min"),
        mes_estrutura=(
            "mes_estrutura",
            "min",
        ),
        TRANSPORTISTA=(
            "TRANSPORTISTA",
            "first",
        ),
        LOCORIGEN=(
            "LOCORIGEN",
            "first",
        ),
        LOCDESTINO=(
            "LOCDESTINO",
            "first",
        ),
        paletes=("PALETS", "sum"),
        paletes_ocupacao=(
            "paletes_ocupacao_codeut",
            "first",
        ),
        peso_kg=("PESO_BRUTO", "sum"),
        custo_transporte=(
            "COSTEDT",
            "sum",
        ),
        custo_estrutura=(
            "custo_estrutura",
            "sum",
        ),
        custo_total=(
            "custo_total_c",
            "sum",
        ),
        ingresso_total=(
            "total_ingresso_c",
            "sum",
        ),
        capacidade_original=(
            "CAMION_CAPACIDAD_NUM",
            moda_segura,
        ),
        capacidade_real=(
            "capacidade_norm",
            moda_segura,
        ),
    )
)

# ==========================
# 2. Estado ocupação
# ==========================
condicoes_ocupacao = [
    df_viagens["capacidade_real"]
    .isna()
    .to_numpy(dtype=bool),

    df_viagens["capacidade_real"]
    .le(0)
    .fillna(False)
    .to_numpy(dtype=bool),

    df_viagens["paletes_ocupacao"]
    .isna()
    .to_numpy(dtype=bool),
]

df_viagens["ocupacao_estado"] = (
    np.select(
        condicoes_ocupacao,
        [
            "SEM_CAPACIDADE",
            "CAPACIDADE_INVALIDA",
            "SEM_PALETES",
        ],
        default="OK",
    )
)

# ==========================
# 3. Ocupação
# ==========================
df_viagens["ocupacao_pct"] = (
    dividir_seguro(
        df_viagens["paletes_ocupacao"],
        df_viagens["capacidade_real"],
    )
    * 100
)

df_viagens.loc[
    ~df_viagens[
        "ocupacao_estado"
    ].eq("OK"),
    "ocupacao_pct",
] = np.nan

# ==========================
# 4. KPIs viagem
# ==========================
df_viagens["peso_ton"] = (
    df_viagens["peso_kg"]
    / 1000
)

df_viagens["custo_por_palete"] = (
    dividir_seguro(
        df_viagens["custo_total"],
        df_viagens["paletes"],
    )
)

df_viagens[
    "ingresso_por_palete"
] = dividir_seguro(
    df_viagens["ingresso_total"],
    df_viagens["paletes"],
)

df_viagens["custo_por_ton"] = (
    dividir_seguro(
        df_viagens["custo_total"],
        df_viagens["peso_ton"],
    )
)

df_viagens[
    "ingresso_por_ton"
] = dividir_seguro(
    df_viagens["ingresso_total"],
    df_viagens["peso_ton"],
)

df_viagens["margem"] = (
    df_viagens["ingresso_total"]
    - df_viagens["custo_total"]
)

df_viagens["margem_pct"] = (
    dividir_seguro(
        df_viagens["margem"],
        df_viagens["ingresso_total"],
    )
    * 100
)

# ==========================
# 5. Associar ao detalhe
# ==========================
df = df.merge(
    df_viagens[
        [
            "CODEUT",
            "capacidade_real",
            "ocupacao_pct",
            "ocupacao_estado",
        ]
    ],
    on="CODEUT",
    how="left",
    validate="many_to_one",
)


## 10. Validações finais

In [48]:
# ==========================
# 1. Estrutura mensal
# ==========================
validacao_estrutura = (
    df.loc[
        df["mes_estrutura"].isin(
            n_codeut_mes.index
        )
    ]
    .groupby(
        "mes_estrutura",
        as_index=False,
    )
    .agg(
        viagens=(
            "CODEUT",
            "nunique",
        ),
        estrutura=(
            "custo_estrutura",
            "sum",
        ),
    )
)

validacao_estrutura["esperado"] = (
    CUSTO_ESTRUTURA_MENSAL
)

validacao_estrutura["erro"] = (
    validacao_estrutura["estrutura"]
    - validacao_estrutura["esperado"]
)

erro_estrutura = (
    validacao_estrutura["erro"]
    .abs()
    .max()
)

if pd.isna(erro_estrutura):
    erro_estrutura = 0.0

# ==========================
# 2. Consistência detalhe/viagens
# ==========================
erro_custo = abs(
    df["custo_total_c"].sum()
    - df_viagens["custo_total"].sum()
)

erro_ingresso = abs(
    df["total_ingresso_c"].sum()
    - df_viagens["ingresso_total"].sum()
)

erro_margem = (
    (
        df["margem"]
        - (
            df["total_ingresso_c"]
            - df["custo_total_c"]
        )
    )
    .abs()
    .max()
)

if pd.isna(erro_margem):
    erro_margem = 0.0

# ==========================
# 3. Validações adicionais
# ==========================
taxas_combustivel_em_falta = int(
    df["taxa_combustivel_em_falta"]
    .sum()
)

linhas_fora_ano = int(
    (
        df["FENTREGA"].notna()
        & ~df["FENTREGA"]
        .dt.year
        .eq(ANO)
    ).sum()
)

# ==========================
# 4. Controlos finais
# ==========================
validacao_financeira = pd.DataFrame({
    "controlo": [
        "Danone df = ASSOCIADO_EXACTO",
        "Danone no troço final",
        "Taxa combustível em falta",
        "FENTREGA fora do ano",
        "Estrutura mensal",
        "Custo detalhe = viagens",
        "Ingresso detalhe = viagens",
        "CODEUT texto nan",
        "CODEUT único em viagens",
        "Ocupação OK > 100%",
        "Margem consistente",
    ],
    "valor": [
        abs(float(erro_reconciliacao_danone)),
        int(casos_troco_danone_errados),
        int(taxas_combustivel_em_falta),
        int(linhas_fora_ano),
        float(erro_estrutura),
        float(erro_custo),
        float(erro_ingresso),
        int(df["CODEUT"].eq("nan").sum()),
        int(
            df_viagens["CODEUT"]
            .duplicated()
            .sum()
        ),
        int(
            df_viagens.loc[
                df_viagens[
                    "ocupacao_estado"
                ].eq("OK"),
                "ocupacao_pct",
            ]
            .gt(100 + 1e-9)
            .sum()
        ),
        float(erro_margem),
    ],
    "tolerancia": [
        TOLERANCIA_FINANCEIRA,
        0,
        0,
        0,
        TOLERANCIA_FINANCEIRA,
        TOLERANCIA_FINANCEIRA,
        TOLERANCIA_FINANCEIRA,
        0,
        0,
        0,
        TOLERANCIA_FINANCEIRA,
    ],
})

validacao_financeira["estado"] = (
    np.where(
        validacao_financeira["valor"]
        <= validacao_financeira[
            "tolerancia"
        ],
        "OK",
        "ERRO",
    )
)

validacao_financeira


,controlo,valor,tolerancia,estado
0,Danone df = ASSOCIADO_EXACTO,2.910383e-11,0.01,OK
1,Danone no troço final,0.000000e+00,0.00,OK
2,Taxa combustível em falta,0.000000e+00,0.00,OK
3,FENTREGA fora do ano,0.000000e+00,0.00,OK
4,Estrutura mensal,1.455192e-11,0.01,OK
5,Custo detalhe = viagens,4.656613e-10,0.01,OK
6,Ingresso detalhe = viagens,4.656613e-10,0.01,OK
7,CODEUT texto nan,0.000000e+00,0.00,OK
8,CODEUT único em viagens,0.000000e+00,0.00,OK
9,Ocupação OK > 100%,1.742000e+03,0.00,ERRO


## 11. Dataset final e exportação

In [49]:
# ==========================
# 1. Colunas finais
# ==========================
colunas_finais = [
    "REFERENCIA",
    "referencia_chave",
    "FENTREGA",
    "FCARGA",
    "CODEUT",
    "capacidade_norm",
    "paletes_codeut",
    "paletes_ocupacao_codeut",
    "capacidade_real",
    "ocupacao_pct",
    "ocupacao_estado",
    "PALETS",
    "PESO_BRUTO",
    "INGRESODT",
    "ingresso_danone",
    "total_ingresso",
    "taxa_combustivel",
    "taxa_combustivel_esperada",
    "taxa_combustivel_em_falta",
    "base_combustivel",
    "acerto_combustivel",
    "total_ingresso_c",
    "COSTEDT",
    "custo_estrutura",
    "custo_total_c",
    "margem",
    "margem_pct",
    "custo_por_palete",
    "ingresso_por_palete",
    "margem_por_palete",
    "PROV_ORIGEN",
    "LOCORIGEN",
    "PROV_DESTINO",
    "LOCDESTINO",
    "CPOSTAL",
    "CPOSTAD",
    "longitude_origem",
    "latitude_origem",
    "longitude_destino",
    "latitude_destino",
    "TRANSPORTISTA",
    "week_day",
    "week_number",
    "ano",
    "mes",
    "mes_nome",
    "data",
    "mes_estrutura",
    "dados_validos",
    "LOCCAR",
    "LUGARCARGA",
    "LOCDES",
    "LUGARDESCARGA",
    "TRACTORA",
    "CODEDT",
    "CODACT",
    "TIPOCLIENTE",
    "TIPOFLUJO",
    "PROV_ENTREGAR",
    "PAISENTREGAR",
    "ACTIVIDAD",
]

colunas_existentes = [
    coluna
    for coluna in colunas_finais
    if coluna in df.columns
]

df_final = (
    df[colunas_existentes]
    .copy()
)

df_final.head()


,REFERENCIA,referencia_chave,FENTREGA,FCARGA,CODEUT,capacidade_norm,paletes_codeut,paletes_ocupacao_codeut,capacidade_real,ocupacao_pct,...,LOCDES,LUGARDESCARGA,TRACTORA,CODEDT,CODACT,TIPOCLIENTE,TIPOFLUJO,PROV_ENTREGAR,PAISENTREGAR,ACTIVIDAD
0,5022105715 414339918,5022105715,2026-03-03,2026-03-02,3387903,33.0,35.0,35.0,33.0,106.060606,...,579957,CMTIR,AO-00-DT,25648132,11,TLD PORTUGAL,Distribucion,Porto,PORTUGAL,DANONE PORTUGAL
1,5022096558 414338874,5022096558,2026-03-03,2026-03-02,3387903,33.0,35.0,35.0,33.0,106.060606,...,579957,CMTIR,AO-00-DT,25648134,11,TLD PORTUGAL,Distribucion,Braga,PORTUGAL,DANONE PORTUGAL
2,5022096556 414340096,5022096556,2026-03-03,2026-03-02,3387903,33.0,35.0,35.0,33.0,106.060606,...,579957,CMTIR,AO-00-DT,25648136,11,TLD PORTUGAL,Distribucion,Porto,PORTUGAL,DANONE PORTUGAL
3,5022149182 414342129,5022149182,2026-03-03,2026-03-02,3387903,33.0,35.0,35.0,33.0,106.060606,...,579957,CMTIR,AO-00-DT,25648138,11,TLD PORTUGAL,Distribucion,Aveiro,PORTUGAL,DANONE PORTUGAL
4,5022149183 414342131,5022149183,2026-03-03,2026-03-02,3387903,33.0,35.0,35.0,33.0,106.060606,...,579957,CMTIR,AO-00-DT,25648140,11,TLD PORTUGAL,Distribucion,Porto,PORTUGAL,DANONE PORTUGAL


In [55]:

# ==========================
# 1. Validar Danone no output
# ==========================
validacao = df.loc[
    df["mes"].eq(9)
].groupby("ACTIVIDAD", dropna=False).agg(
    ingresso_danone=("ingresso_danone", "sum"),
    ingresso_total=("total_ingresso", "sum"),
).reset_index()

display(
    validacao.loc[
        validacao["ACTIVIDAD"].astype("string").str.contains(
            "DANONE", case=False, na=False
        )
    ]
)


,ACTIVIDAD,ingresso_danone,ingresso_total
11,DANONE PORTUGAL,156349.618005,156349.618005
12,DANONE PORTUGAL TLD,0.0,567.88


In [50]:
# ==========================
# 1. Exportar detalhe
# ==========================
df_final.to_parquet(
    PATHS["parquet"],
    index=False,
)

# ==========================
# 2. Exportar viagens
# ==========================
df_viagens.to_parquet(
    PATHS["viagens"],
    index=False,
)

# ==========================
# 3. Exportar validações
# ==========================
validacao_financeira.to_csv(
    PATHS["validacao"],
    index=False,
    sep=";",
    decimal=",",
)

validacao_danone.to_csv(
    PATHS["danone_diag"],
    index=False,
    sep=";",
    decimal=",",
)

# ==========================
# 4. Resultado
# ==========================
pd.DataFrame({
    "output": [
        "Detalhe Power BI",
        "Viagens",
    ],
    "ficheiro": [
        str(PATHS["parquet"]),
        str(PATHS["viagens"]),
    ],
    "linhas": [
        len(df_final),
        len(df_viagens),
    ],
})

,output,ficheiro,linhas
0,Detalhe Power BI,C:\Users\LISARR\OneDrive - Salvesen Logística ...,98239
1,Viagens,C:\Users\LISARR\OneDrive - Salvesen Logística ...,15585


# 12. Análises

A partir deste ponto não são alteradas regras de negócio nem valores do pipeline.  
As análises usam apenas `df_final`, `df_viagens` e as tabelas de validação já produzidas.

In [51]:
# ==========================
# 1. Resumo financeiro mensal
# ==========================
resumo_mensal = (
    df_final
    .groupby(
        "mes_estrutura",
        as_index=False,
    )
    .agg(
        linhas=("CODEUT", "size"),
        viagens=("CODEUT", "nunique"),
        paletes=("PALETS", "sum"),
        peso_kg=("PESO_BRUTO", "sum"),
        ingresso=(
            "total_ingresso_c",
            "sum",
        ),
        custo=(
            "custo_total_c",
            "sum",
        ),
        margem=("margem", "sum"),
        estrutura=(
            "custo_estrutura",
            "sum",
        ),
        combustivel=(
            "acerto_combustivel",
            "sum",
        ),
    )
)

resumo_mensal["peso_ton"] = (
    resumo_mensal["peso_kg"]
    / 1000
)

resumo_mensal["margem_pct"] = (
    dividir_seguro(
        resumo_mensal["margem"],
        resumo_mensal["ingresso"],
    )
    * 100
)

resumo_mensal

,mes_estrutura,linhas,viagens,paletes,peso_kg,ingresso,custo,margem,estrutura,combustivel,peso_ton,margem_pct
0,2026-01-01,1374,425,3912.0,707621.721,0.0,103000.0,-103000.0,103000.0,0.000000,707.621721,<NA>
1,2026-02-01,1138,382,2846.0,458832.827,0.0,103000.0,-103000.0,103000.0,0.000000,458.832827,<NA>
2,2026-03-01,12281,1966,46487.563,12553858.055,361568.980706,540931.2,-179362.219294,103000.0,2926.743206,12553.858055,-49.606639
3,2026-04-01,12577,1972,48077.75,11656719.178,400222.722927,572365.89,-172143.167073,103000.0,20632.548427,11656.719178,-43.011842
4,2026-05-01,12581,1953,47682.05,11801330.019,401786.425876,573870.16,-172083.734124,103000.0,30278.589876,11801.330019,-42.829654
5,2026-06-01,12460,1966,45447.16,11952113.721,380567.972446,580716.09,-200148.117554,103000.0,26496.304446,11952.113721,-52.59195
6,2026-07-01,14179,2167,49278.5,13429174.515,423905.369376,643897.92,-219992.550624,103000.0,21443.142376,13429.174515,-51.896618
7,2026-08-01,13273,2019,49250.0,12551971.754,424690.680698,607085.81,-182395.129302,103000.0,24252.083698,12551.971754,-42.947759
8,2026-09-01,13534,2026,47305.987,12114236.275,229174.797537,159363.7,69811.097537,103000.0,20391.661532,12114.236275,30.461944
9,2026-10-01,4842,709,14040.98,3558349.334,0.0,103000.0,-103000.0,103000.0,0.000000,3558.349334,<NA>


In [52]:
# ==========================
# 1. KPIs de viagens
# ==========================
viagens_validas = (
    df_viagens.loc[
        df_viagens[
            "ocupacao_estado"
        ].eq("OK")
    ]
    .copy()
)

kpi_viagens = pd.DataFrame({
    "metrica": [
        "Viagens",
        "Paletes",
        "Peso ton",
        "Ocupação média %",
        "Custo por viagem",
        "Custo por palete",
        "Custo por ton",
        "Margem",
    ],
    "valor": [
        df_viagens["CODEUT"].nunique(),
        df_viagens["paletes"].sum(),
        df_viagens["peso_ton"].sum(),
        viagens_validas[
            "ocupacao_pct"
        ].mean(),
        df_viagens[
            "custo_total"
        ].mean(),
        dividir_seguro(
            pd.Series([
                df_viagens[
                    "custo_total"
                ].sum()
            ]),
            pd.Series([
                df_viagens[
                    "paletes"
                ].sum()
            ]),
        ).iloc[0],
        dividir_seguro(
            pd.Series([
                df_viagens[
                    "custo_total"
                ].sum()
            ]),
            pd.Series([
                df_viagens[
                    "peso_ton"
                ].sum()
            ]),
        ).iloc[0],
        df_viagens["margem"].sum(),
    ],
})

kpi_viagens

,metrica,valor
0,Viagens,1.558500e+04
1,Paletes,3.543280e+05
2,Peso ton,9.078421e+04
3,Ocupação média %,7.468538e+01
4,Custo por viagem,2.558377e+02
5,Custo por palete,1.125294e+01
6,Custo por ton,4.391987e+01
7,Margem,-1.365314e+06


In [53]:

# ==========================
# 1. Viagens únicas
# ==========================

validacao_enchimento = (
    df.loc[
        df["dados_validos"]
        & df["TRANSPORTISTA"].notna()
    ]
    .groupby(
        "CODEUT",
        as_index=False,
    )
    .agg(
        transportador=(
            "TRANSPORTISTA",
            "first",
        ),
        paletes=(
            "paletes_codeut",
            "first",
        ),
        paletes_ocupacao=(
            "paletes_ocupacao_codeut",
            "first",
        ),
        capacidade=(
            "capacidade_norm",
            "first",
        ),
    )
)


# ==========================
# 2. Taxa por viagem
# ==========================

validacao_enchimento["taxa_enchimento"] = (
    validacao_enchimento["paletes_ocupacao"]
    / validacao_enchimento["capacidade"]
)

validacao_enchimento["taxa_enchimento_pct"] = (
    validacao_enchimento["taxa_enchimento"]
    * 100
)


# ==========================
# 3. Resumo por transportador
# ==========================

enchimento_transportador = (
    validacao_enchimento
    .groupby(
        "transportador",
        as_index=False,
    )
    .agg(
        viagens=(
            "CODEUT",
            "nunique",
        ),
        paletes=(
            "paletes_ocupacao",
            "sum",
        ),
        capacidade_total=(
            "capacidade",
            "sum",
        ),
        taxa_media_viagem_pct=(
            "taxa_enchimento_pct",
            "mean",
        ),
    )
)


# ==========================
# 4. Taxa ponderada
# ==========================

enchimento_transportador[
    "taxa_enchimento_pct"
] = (
    enchimento_transportador["paletes"]
    / enchimento_transportador["capacidade_total"]
    * 100
)


# ==========================
# 5. Formatar
# ==========================

enchimento_transportador = (
    enchimento_transportador
    .sort_values(
        "taxa_enchimento_pct",
        ascending=False,
    )
    .reset_index(drop=True)
)

enchimento_transportador[
    [
        "taxa_media_viagem_pct",
        "taxa_enchimento_pct",
    ]
] = (
    enchimento_transportador[
        [
            "taxa_media_viagem_pct",
            "taxa_enchimento_pct",
        ]
    ]
    .round(2)
)


# ==========================
# 6. Resultado
# ==========================

enchimento_transportador

,transportador,viagens,paletes,capacidade_total,taxa_media_viagem_pct,taxa_enchimento_pct
0,EL MOSCA S.A.,4,132.0,132.0,100.0,100.0
1,MOSCA PORTUGAL LDA,64,2112.0,2112.0,100.0,100.0
2,NO USAR,9,297.0,297.0,100.0,100.0
3,LOGISTA FREIGHT S.A.U.,10,330.0,330.0,100.0,100.0
4,SALVESEN,160,8461.25,8809.0,98.78,96.05
5,TORRESTIR - TRANSPORTES NACIONAIS E INTERNACIO...,128,6192.0,6534.0,100.03,94.77
6,JMR PRESTAÇAO DE SERVIÇOS PARA A DISTRIBUÇAO S.A,1439,46646.0,54994.0,85.18,84.82
7,MODELO CONTINENTE HIPERMERCADOS S.A.,1009,30622.73,36505.0,83.73,83.89
8,TRANSPORTES MACHADO & BRITES LDA,4,71.0,85.0,98.02,83.53
9,TRANSPORTES PAULO COSTA & FERREIRA LDA,360,11427.0,13753.0,83.0,83.09


In [54]:
# ==========================
# 7. Média global
# ==========================

taxa_enchimento_global_pct = (
    validacao_enchimento["paletes_ocupacao"].sum()
    / validacao_enchimento["capacidade"].sum()
    * 100
)

taxa_enchimento_global_pct

np.float64(74.9778243939141)